# Deliverable 3: Ultrasound Physics & Imaging


---

## Student Information

**Name:** [Your Name]  
**UCID:** [Your UCID]  
**Date:** [Submission Date]

---

## Overview

In this deliverable, you will apply ultrasound physics principles to design algorithms for:
1. **Part 1**: Analyzing acoustic transmission through tissue layers
2. **Part 2**: Measuring blood velocity with Doppler ultrasound
3. **Part 3**: Quantifying resolution and classifying artifacts
4. **Part 4**: Integrating knowledge in a clinical memo

**Estimated Time:** 6 hours

---

## Setup and Imports

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq
from pathlib import Path

# Local utilities
import sys
sys.path.insert(0, 'data')
from us_utils import (
    load_tissue_properties,
    load_clinical_scenarios,
    load_doppler_signal,
    list_doppler_signals,
    load_phantom_data,
    load_artifact_image,
    SPEED_OF_SOUND_TISSUE,
    calculate_wavelength,
    max_unambiguous_velocity,
    max_unambiguous_depth
)

# Plot settings
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True

# Data directory
DATA_DIR = Path('data')

print("Setup complete!")

In [ ]:
# Load data files
tissue_props = load_tissue_properties(DATA_DIR)
clinical_scenarios = load_clinical_scenarios(DATA_DIR)

# Display tissue properties
print("Tissue Acoustic Properties:")
print(tissue_props.to_string(index=False))

---

# Part 1: Acoustic Impedance & Transmission Analyzer (30 points)

Design an algorithm that analyzes ultrasound transmission through layered tissue structures.

---

## 1.1 Acoustic Impedance Calculator (6 points)

Implement a function to calculate acoustic impedance from tissue properties.

**Key Equation:**
$$Z = \rho \cdot c$$

where:
- $Z$ = acoustic impedance (Rayl = kg/(m²·s))
- $\rho$ = density (kg/m³)
- $c$ = speed of sound (m/s)

In [ ]:
def calculate_acoustic_impedance(density_kg_m3, speed_of_sound_m_s):
    """
    Calculate acoustic impedance.
    
    Parameters
    ----------
    density_kg_m3 : float or array
        Tissue density in kg/m³
    speed_of_sound_m_s : float or array
        Speed of sound in m/s
    
    Returns
    -------
    float or array
        Acoustic impedance in MRayl (10^6 Rayl)
    """
    # TODO: Implement impedance calculation
    # Return impedance in MRayl for convenient display
    pass

In [ ]:
# Test your impedance calculator
# Calculate impedance for all tissues and compare with provided values

# TODO: Apply your function to the tissue_props dataframe
# Compare your calculated values with the 'acoustic_impedance_MRayl' column

## 1.2 Reflection and Transmission Coefficients (8 points)

Implement functions to calculate reflection and transmission coefficients at a tissue interface.

**Key Equations (Normal Incidence):**

Pressure reflection coefficient:
$$R = \frac{Z_2 - Z_1}{Z_2 + Z_1}$$

Pressure transmission coefficient:
$$T = \frac{2Z_2}{Z_2 + Z_1}$$

**Energy Conservation:**
$$R_I + T_I = 1$$

where $R_I = R^2$ and $T_I = \frac{4Z_1Z_2}{(Z_1+Z_2)^2}$

In [ ]:
def calculate_reflection_coefficient(Z1, Z2):
    """
    Calculate pressure reflection coefficient at normal incidence.
    
    Parameters
    ----------
    Z1 : float
        Acoustic impedance of medium 1 (incident side)
    Z2 : float
        Acoustic impedance of medium 2 (transmitted side)
    
    Returns
    -------
    float
        Pressure reflection coefficient (can be negative)
    """
    # TODO: Implement reflection coefficient
    pass


def calculate_transmission_coefficient(Z1, Z2):
    """
    Calculate pressure transmission coefficient at normal incidence.
    
    Parameters
    ----------
    Z1 : float
        Acoustic impedance of medium 1 (incident side)
    Z2 : float
        Acoustic impedance of medium 2 (transmitted side)
    
    Returns
    -------
    float
        Pressure transmission coefficient (always positive)
    """
    # TODO: Implement transmission coefficient
    pass


def verify_energy_conservation(Z1, Z2):
    """
    Verify that energy is conserved at an interface.
    
    Returns
    -------
    dict
        Dictionary with R_intensity, T_intensity, and their sum
    """
    # TODO: Calculate intensity coefficients and verify they sum to 1
    pass

In [ ]:
# Test your reflection/transmission functions
# Example: Calculate R and T for soft tissue to bone interface

# TODO: Test with several tissue pairs
# - Soft tissue to bone (high reflection)
# - Fat to liver (low reflection)
# - Tissue to air (nearly total reflection)

## 1.3 Multi-Layer Transmission Model (10 points)

Design an algorithm that calculates the total transmission through multiple tissue layers, accounting for:
- Reflection at each interface
- Attenuation within each layer

**Attenuation Model:**
$$I(x) = I_0 \cdot e^{-2\alpha x}$$

where:
- $\alpha$ = attenuation coefficient (Np/cm)
- The factor of 2 accounts for round-trip travel

**Conversion:** $\alpha_{Np/cm} = \alpha_{dB/cm} \cdot \frac{\ln(10)}{20} \approx \alpha_{dB/cm} \times 0.115$

In [ ]:
def calculate_multilayer_transmission(layers, thicknesses_cm, frequency_mhz, tissue_props_df):
    """
    Calculate total transmission through multiple tissue layers.
    
    Parameters
    ----------
    layers : list of str
        List of tissue names in order (e.g., ['skin', 'fat', 'muscle', 'liver'])
    thicknesses_cm : list of float
        Thickness of each layer in cm
    frequency_mhz : float
        Ultrasound frequency in MHz
    tissue_props_df : pd.DataFrame
        DataFrame with tissue properties
    
    Returns
    -------
    dict
        Dictionary with:
        - 'total_transmission_intensity': fraction of intensity reaching target
        - 'interface_losses': list of transmission at each interface
        - 'attenuation_losses': list of attenuation in each layer (dB)
        - 'cumulative_depth': depth to each interface
    """
    # TODO: Implement multi-layer transmission model
    # 1. For each interface, calculate transmission coefficient
    # 2. For each layer, calculate attenuation
    # 3. Combine all losses to get total transmission
    pass

In [ ]:
# Test with a clinical scenario
scenario = clinical_scenarios.iloc[0]  # Abdominal liver imaging
print(f"Scenario: {scenario['name']}")
print(f"Layers: {scenario['layers']}")
print(f"Thicknesses: {scenario['layer_thicknesses_cm']} cm")
print(f"Frequency: {scenario['frequency_MHz']} MHz")

# TODO: Apply your multilayer transmission function
# Parse the layers and thicknesses from the scenario

## 1.4 Clinical Scenario Analysis (6 points)

Apply your transmission model to analyze all clinical scenarios. For each scenario:
1. Calculate total transmission to the target
2. Identify the dominant source of signal loss
3. Make recommendations for imaging optimization

In [ ]:
def analyze_clinical_scenarios(scenarios_df, tissue_props_df):
    """
    Analyze all clinical scenarios for transmission efficiency.
    
    Returns
    -------
    pd.DataFrame
        Results with transmission analysis for each scenario
    """
    # TODO: Analyze all scenarios
    # For each scenario:
    # 1. Calculate transmission
    # 2. Identify limiting factors
    # 3. Suggest optimizations
    pass

In [ ]:
# TODO: Run analysis and create visualization
# Create a figure showing transmission vs depth for different scenarios

### Question 1.4.1 (2 points)

Which clinical scenario has the lowest transmission efficiency? Explain why in terms of the physics involved.

**Your Answer:**

[Write your analysis here]

### Question 1.4.2 (2 points)

For the thyroid examination scenario, would you recommend increasing or decreasing the frequency? Justify your answer considering both resolution and penetration.

**Your Answer:**

[Write your analysis here]

---

# Part 2: Doppler Velocity Measurement System (30 points)

Design a system that extracts blood velocity from Doppler signals and handles aliasing.

---

## 2.1 Doppler Velocity Extraction (8 points)

Implement an algorithm to extract blood velocity from a Doppler signal.

**Key Equation:**
$$f_D = \frac{2 v f_0 \cos\theta}{c}$$

Rearranging for velocity:
$$v = \frac{f_D \cdot c}{2 f_0 \cos\theta}$$

In [ ]:
# Explore available Doppler signals
available_signals = list_doppler_signals(DATA_DIR / 'doppler_signals')
print("Available Doppler signals:")
for sig in sorted(available_signals):
    print(f"  - {sig}")

In [ ]:
# Load and examine a sample signal
sample_signal = load_doppler_signal('carotid_normal', DATA_DIR / 'doppler_signals')

print("Signal properties:")
print(f"  Sampling frequency (PRF): {sample_signal['prf']} Hz")
print(f"  Center frequency: {sample_signal['f0']/1e6:.1f} MHz")
print(f"  Doppler angle: {sample_signal['angle_deg']}°")
print(f"  Depth: {sample_signal['depth']*100:.1f} cm")
print(f"  Number of samples: {len(sample_signal['signal'])}")

In [ ]:
def extract_doppler_frequency(signal_data):
    """
    Extract dominant Doppler frequency shift from signal.
    
    Parameters
    ----------
    signal_data : dict
        Dictionary with 'signal', 'fs' (same as PRF), and other metadata
    
    Returns
    -------
    float
        Dominant Doppler frequency shift in Hz
    """
    # TODO: Implement FFT-based frequency detection
    # 1. Compute FFT of the signal
    # 2. Find the peak frequency
    # 3. Return the Doppler shift
    pass


def calculate_velocity_from_doppler(f_doppler, f0, theta_deg, c=1540):
    """
    Calculate blood velocity from Doppler frequency shift.
    
    Parameters
    ----------
    f_doppler : float
        Doppler frequency shift in Hz
    f0 : float
        Transducer center frequency in Hz
    theta_deg : float
        Doppler angle in degrees
    c : float
        Speed of sound in m/s
    
    Returns
    -------
    float
        Blood velocity in m/s
    """
    # TODO: Implement velocity calculation from Doppler equation
    pass

In [ ]:
# Test your velocity extraction on the sample signal

# TODO: Extract velocity and compare with true value in metadata
# The true velocity is stored in sample_signal['metadata']['true_velocity_m_s']

## 2.2 Aliasing Detection (8 points)

Design an algorithm to detect when a Doppler signal is aliased.

**Nyquist Criterion:**
$$v_{max} = \frac{c \cdot PRF}{4 f_0 \cos\theta}$$

Aliasing occurs when $|v| > v_{max}$

In [ ]:
def detect_aliasing(signal_data):
    """
    Detect if a Doppler signal is likely aliased.
    
    Parameters
    ----------
    signal_data : dict
        Doppler signal data dictionary
    
    Returns
    -------
    dict
        Dictionary with:
        - 'is_aliased': bool
        - 'measured_velocity': extracted velocity
        - 'nyquist_velocity': maximum unambiguous velocity
        - 'confidence': confidence in aliasing detection
    """
    # TODO: Implement aliasing detection
    # Consider:
    # - Frequency components near Nyquist limit
    # - Sign reversals in the spectrum
    # - Spectral wrap-around characteristics
    pass

In [ ]:
# Test aliasing detection on various signals
test_signals = ['carotid_normal', 'carotid_stenosis', 'aortic_stenosis', 'test_low_prf', 'test_high_prf']

# TODO: Test your aliasing detector on these signals
# Compare with the ground truth in metadata['is_aliased']

## 2.3 PRF Optimization Algorithm (8 points)

Design an algorithm that selects the optimal PRF given:
- Target depth (constrains maximum PRF)
- Expected velocity range (constrains minimum PRF to avoid aliasing)

**Constraints:**
$$PRF_{max} = \frac{c}{2 \cdot d_{max}}$$
$$PRF_{min} = \frac{4 f_0 v_{max} \cos\theta}{c}$$

In [ ]:
def optimize_prf(depth_m, expected_velocity_m_s, f0_hz, theta_deg=60, c=1540):
    """
    Calculate optimal PRF for Doppler measurement.
    
    Parameters
    ----------
    depth_m : float
        Target depth in meters
    expected_velocity_m_s : float
        Maximum expected velocity in m/s
    f0_hz : float
        Transducer center frequency in Hz
    theta_deg : float
        Doppler angle in degrees
    c : float
        Speed of sound in m/s
    
    Returns
    -------
    dict
        Dictionary with:
        - 'optimal_prf': recommended PRF
        - 'prf_min': minimum PRF to avoid aliasing
        - 'prf_max': maximum PRF for depth
        - 'is_feasible': whether both constraints can be satisfied
        - 'recommendations': list of suggestions if not feasible
    """
    # TODO: Implement PRF optimization
    # If constraints conflict, provide recommendations:
    # - Lower frequency transducer
    # - Smaller Doppler angle
    # - Accept some aliasing
    pass

In [ ]:
# Test PRF optimization for different clinical scenarios

# Scenario 1: Superficial vessel (easy case)
# TODO: depth=0.02m, velocity=1.0 m/s, f0=10 MHz

# Scenario 2: Deep vessel (challenging case)
# TODO: depth=0.15m, velocity=1.5 m/s, f0=2 MHz

# Scenario 3: High velocity at moderate depth (may be infeasible)
# TODO: depth=0.08m, velocity=4.0 m/s, f0=5 MHz

## 2.4 Clinical Doppler Analysis (6 points)

Apply your Doppler analysis system to all available signals and generate a clinical report.

In [ ]:
def analyze_all_doppler_signals(data_dir):
    """
    Analyze all Doppler signals and generate report.
    
    Returns
    -------
    pd.DataFrame
        Analysis results for all signals
    """
    # TODO: Analyze all signals
    # For each signal:
    # 1. Extract velocity
    # 2. Check for aliasing
    # 3. Calculate error vs ground truth
    # 4. Recommend PRF adjustments if needed
    pass

In [ ]:
# TODO: Run analysis and visualize results
# Create figure showing:
# 1. Measured vs true velocity
# 2. Aliased signals highlighted
# 3. PRF optimization suggestions

### Question 2.4.1 (3 points)

For the `aortic_stenosis` signal, the velocity is very high. What strategies could a sonographer use to accurately measure this velocity?

**Your Answer:**

[Write your analysis here]

### Question 2.4.2 (3 points)

Explain why continuous-wave (CW) Doppler can measure higher velocities than pulsed-wave (PW) Doppler. What trade-off does CW Doppler make to achieve this?

**Your Answer:**

[Write your analysis here]

---

# Part 3: Resolution & Artifact Analyzer (30 points)

Design algorithms to measure image resolution and classify common ultrasound artifacts.

---

## 3.1 Resolution Measurement (10 points)

Measure axial and lateral resolution from wire phantom data.

**Resolution Definition:** Full Width at Half Maximum (FWHM) of the point spread function.

**Theoretical Relationships:**
- Axial resolution: $AR = \frac{N_c \cdot \lambda}{2}$ (depends on pulse length)
- Lateral resolution: $LR = \frac{\lambda \cdot F}{L}$ (depends on focusing)

In [ ]:
# Load wire phantom data
phantom_data = load_phantom_data(DATA_DIR)

print("Phantom data:")
print(f"  Image shape: {phantom_data['image'].shape}")
print(f"  Pixel size: {phantom_data['pixel_size_mm']} mm")
print(f"  Frequency: {phantom_data['frequency_mhz']} MHz")
print(f"  Focal depth: {phantom_data['focal_depth_mm']} mm")
print(f"\nWire positions (lateral, axial) in mm:")
for pos in phantom_data['wire_positions_mm']:
    print(f"  {pos}")

In [ ]:
# Display phantom image
plt.figure(figsize=(8, 10))
extent = [0, phantom_data['image'].shape[1] * phantom_data['pixel_size_mm'],
          phantom_data['image'].shape[0] * phantom_data['pixel_size_mm'], 0]
plt.imshow(phantom_data['image'], cmap='gray', extent=extent)
plt.colorbar(label='dB')
plt.xlabel('Lateral (mm)')
plt.ylabel('Depth (mm)')
plt.title('Wire Phantom Image')

# Mark wire positions
for lat, ax in phantom_data['wire_positions_mm']:
    plt.plot(lat, ax, 'r+', markersize=15, markeredgewidth=2)

plt.tight_layout()
plt.show()

In [ ]:
def measure_fwhm(profile):
    """
    Measure Full Width at Half Maximum of a 1D profile.
    
    Parameters
    ----------
    profile : array
        1D intensity profile through PSF
    
    Returns
    -------
    float
        FWHM in pixels
    """
    # TODO: Implement FWHM measurement
    # 1. Find maximum
    # 2. Find half maximum level
    # 3. Find crossing points
    # 4. Return width
    pass


def measure_resolution(image, wire_position_px, pixel_size_mm):
    """
    Measure axial and lateral resolution at a wire position.
    
    Parameters
    ----------
    image : 2D array
        Ultrasound image
    wire_position_px : tuple
        (x, y) position in pixels
    pixel_size_mm : float
        Pixel size in mm
    
    Returns
    -------
    dict
        Dictionary with axial_resolution_mm, lateral_resolution_mm, and profiles
    """
    # TODO: Implement resolution measurement
    # 1. Extract axial profile (vertical through wire)
    # 2. Extract lateral profile (horizontal through wire)
    # 3. Measure FWHM of each
    # 4. Convert to mm
    pass

In [ ]:
# TODO: Measure resolution at each wire position
# Compare axial resolution (should be constant)
# Compare lateral resolution (should vary with depth)

In [ ]:
# TODO: Create visualization showing resolution vs depth
# Plot axial and lateral resolution as function of depth
# Indicate focal zone

## 3.2 Artifact Classification (12 points)

Design a classifier that can distinguish between:
1. **Acoustic Shadowing** - reduced signal behind highly attenuating structures
2. **Posterior Enhancement** - increased signal behind low-attenuation structures
3. **Reverberation** - equally spaced horizontal lines from multiple reflections

In [ ]:
# Load artifact images
shadow_data = load_artifact_image('shadow', DATA_DIR)
enhance_data = load_artifact_image('enhancement', DATA_DIR)
reverb_data = load_artifact_image('reverberation', DATA_DIR)

# Display all three
fig, axes = plt.subplots(1, 3, figsize=(15, 6))

for ax, (data, title) in zip(axes, [
    (shadow_data, 'Acoustic Shadow'),
    (enhance_data, 'Posterior Enhancement'),
    (reverb_data, 'Reverberation')
]):
    ax.imshow(data['image'], cmap='gray')
    ax.set_title(title)
    ax.set_xlabel('Lateral (pixels)')
    ax.set_ylabel('Depth (pixels)')

plt.tight_layout()
plt.show()

In [ ]:
def extract_artifact_features(image):
    """
    Extract features useful for artifact classification.
    
    Parameters
    ----------
    image : 2D array
        Ultrasound image with potential artifact
    
    Returns
    -------
    dict
        Dictionary of features useful for classification
    """
    # TODO: Extract discriminative features
    # Consider:
    # - Vertical intensity gradients (shadow vs enhancement)
    # - Horizontal line detection (reverberation)
    # - Local intensity statistics
    # - Autocorrelation for periodic patterns
    pass


def classify_artifact(image):
    """
    Classify the type of artifact in an ultrasound image.
    
    Parameters
    ----------
    image : 2D array
        Ultrasound image with artifact
    
    Returns
    -------
    dict
        Dictionary with:
        - 'classification': str ('shadow', 'enhancement', 'reverberation')
        - 'confidence': float (0-1)
        - 'features': extracted features
        - 'explanation': why this classification
    """
    # TODO: Implement artifact classifier
    # 1. Extract features
    # 2. Apply classification logic
    # 3. Return result with explanation
    pass

In [ ]:
# TODO: Test your classifier on all three artifact images
# Report classification and confidence for each

## 3.3 Artifact Impact Analysis (8 points)

For each artifact type, analyze:
1. Physical cause
2. Clinical implications
3. Mitigation strategies

In [ ]:
def analyze_artifact_impact(artifact_type, image_data):
    """
    Analyze the clinical impact of an artifact.
    
    Parameters
    ----------
    artifact_type : str
        Type of artifact
    image_data : dict
        Artifact image data
    
    Returns
    -------
    dict
        Analysis including cause, implications, and mitigation
    """
    # TODO: Analyze each artifact type
    pass

In [ ]:
# TODO: Analyze all three artifact types
# Create a summary table or visualization

### Question 3.3.1 (4 points)

A sonographer is imaging a gallbladder and sees a bright echo with a dark region behind it. How can they determine if this represents:
a) A gallstone with acoustic shadowing, or
b) A polyp without shadowing?

What additional imaging maneuvers could help differentiate?

**Your Answer:**

[Write your analysis here]

### Question 3.3.2 (4 points)

Explain why reverberation artifacts appear as equally spaced horizontal lines. What determines the spacing between the lines?

**Your Answer:**

[Write your analysis here]

---

# Part 4: Clinical Integration Memo (10 points)

Write a professional memo to a vascular medicine physician comparing ultrasound with other imaging modalities for evaluating deep vein thrombosis (DVT).

---

## Memo Assignment

**Scenario:** A vascular medicine physician asks you (the imaging physicist) to explain why compression ultrasound with Doppler is the first-line imaging test for suspected DVT, rather than CT venography or MR venography.

**Your memo should address:**

1. **Physics Advantages of Ultrasound for DVT**
   - How acoustic impedance differences enable thrombus visualization
   - How Doppler detects flow abnormalities
   - Real-time compression capability

2. **Technical Limitations to Consider**
   - When ultrasound may be inadequate (e.g., body habitus, calf veins)
   - Doppler limitations (PRF constraints, angle dependence)

3. **When Alternative Modalities May Be Needed**
   - Specific scenarios where CT or MR would be preferred
   - Trade-offs involved

**Format:** Professional medical memo (300-500 words)

## Your Memo

---

**MEMORANDUM**

**TO:** Dr. [Physician Name], Vascular Medicine  
**FROM:** [Your Name], Medical Physics  
**DATE:** [Date]  
**RE:** Imaging Physics Considerations for DVT Evaluation

---

[Write your memo here]

---

---

## Submission Requirements

**You must submit TWO files to D2L:**

1. **Jupyter Notebook** (`.ipynb`): All code, outputs, and analysis. All cells must be executed in order.

2. **PDF Export** (`.pdf`): Exported version of your notebook. Verify all figures and equations render correctly.

**Important:** All answers must be completed directly in this Jupyter notebook.

**File naming convention:** `LastName_FirstName_Deliverable3.ipynb` and `.pdf`

**Due:** Sunday, March 23, 2026 at 11:59 PM

---

## Submission Checklist

Before submitting, verify:

- [ ] All code cells execute without errors
- [ ] All required functions are implemented and documented
- [ ] All visualizations display correctly with labeled axes
- [ ] All questions are answered with appropriate depth
- [ ] Clinical memo is complete and professional
- [ ] File is named: `LastName_FirstName_Deliverable3.ipynb`

---